# DS01 · Arrays, axes, broadcasting, and masks

<!-- paper-first -->
### Begin with the paper question

**Read or revisit [PP01](../../curriculum/papers/processing.md#pp01), [PM05](../../curriculum/papers/modeling.md#pm05).** Use the assigned first-pass sections in the guide; if you already read them, return only to the relevant figure or claim. Do this before the technical explanation below.

**Motivation:** Which axis represents space, time, or a sample, and what changes if AI silently swaps them?

Write a two-sentence prediction and one thing you cannot yet explain. Ask your AI tutor to locate evidence in the supplied paper and distinguish it from inference. A paper link motivates this question; it does not mean the paper uses every method demonstrated here.

**After the experiment:** revisit your prediction in the [evidence ledger](../../curriculum/coursework/EVIDENCE_LEDGER.md). Explain one mechanism you now understand, cite a result from this notebook, and name a paper claim this exercise still cannot test. Keep a small demonstration distinct from a reproduction of the study.
<!-- /paper-first -->

**Format:** 75–100 minutes of guided work, plus 30–60 minutes in the assigned existing course material. **Prerequisite:** the foundations notebooks; follow this strand in order. The core exercise is synthetic, offline, and independently runnable. It demonstrates mechanics, not a validated participant-data analysis.

## Existing course material

Read [Data 8: Arrays](https://github.com/data-8/textbook/blob/5235b7653f8dfaeb90e43419b9aa069322f2d60b/chapters/05/1/Arrays.ipynb). Use the indicated topic, then return here to apply it to a neuroimaging question. Berkeley material is linked in its original form, not adapted or redistributed; its CC BY-NC-ND terms remain upstream. Neuromatch material is CC BY 4.0 with separately licensed software; selected unmodified copies live in `third_party/data_science`. The explanation and dataset below are original.

## Understand the transformation

A neuroimaging array only becomes interpretable when its axes are named. In this lesson the convention is space-x, space-y, space-z, time. A shape of `(6,5,4,10)` is storage information, not anatomical orientation: an affine and acquisition metadata establish physical space and time. The same array shape can represent quite different measurements.

Broadcasting lets one smaller array operate across a larger one. Subtracting a per-voxel temporal mean requires a singleton time axis, so each location gets its own baseline. Subtracting one global mean removes a different component. A mask selects locations; it does not register them. After masking, we use rows for selected voxels and columns for time. Many modeling tools expect observations in rows, so transposition is a scientific decision about what counts as an observation.

An average reduces dimensionality by destroying the variation along an axis. That is useful when the question needs a summary, but a mean image cannot recover temporal responses. A mask can also silently exclude the target tissue. Count selected locations, inspect the selection spatially, and preserve coordinates alongside the reduced matrix. This is the first defense against AI producing valid mathematics for a different question.

## AI-guided prediction

First answer in your own words; then send this to Goose/Ollama or ChatGPT:

> Ask me to predict the shape after subtracting a temporal mean with keepdims=True and after spatial masking. Give at most 15 lines per operation. Explain why transpose changes the observation convention but not the physical image orientation.

Use the model as a tutor and snippet writer. Require it to name the axes, units, fitting population, expected output, and one failure check. A code cell that runs is not proof that it answers the scientific question. Keep raw data unchanged and save your actual settings.

## Experiment

Predict 120 selected-or-unselected spatial locations and 10 samples per location. Apply a mask selecting the first half of x, then compare per-voxel centering with global centering. Deliberate error: flatten all dimensions and claim the result still preserves time.

Run the following cells in order. Before each, predict what should remain unchanged and what should differ. The assertions test specific mathematical or bookkeeping properties, not clinical validity.

In [1]:
import numpy as np
rng = np.random.default_rng(301)
a = rng.normal(size=(6, 5, 4, 10)) + np.arange(6)[:, None, None, None]
centered = a - a.mean(axis=-1, keepdims=True)
mask = np.indices(a.shape[:3])[0] < 3
matrix = centered[mask]
print(a.shape, centered.shape, matrix.shape)
assert matrix.shape == (60, 10)
assert np.allclose(centered.mean(axis=-1), 0)
wrong = a - a.mean()
assert not np.allclose(wrong.mean(axis=-1), 0)
print('Max wrong residual mean:', abs(wrong.mean(axis=-1)).max())

(6, 5, 4, 10) (6, 5, 4, 10) (60, 10)
Max wrong residual mean: 3.5144396331935086


## Explain, break, transfer

1. Save an input → operation → output diagram and state what information was lost.
2. Make the specified wrong choice above. Compare its result with the reference checks; explain why the misleading result is possible.
3. Work through the assigned upstream chapter's example using its own environment or hosted reader. Record one difference between its data and a participant/voxel/time-series dataset.
4. Ask the AI for a short application to a real imaging table, but do not run it until participant identifiers, units, missingness, and any training/test boundary are explicit. Never infer those properties from the column names alone.

**Evidence to submit:** one labeled result, the changed parameter, a failure diagnosis, and a five-sentence interpretation that separates a computational check from the research claim. Explain the result without looking at the model's wording.

<details><summary>Instructor check / answer guide</summary>

Centering over the last axis yields per-voxel means near zero. Masking selects 60 voxels and produces (60,10); global centering generally does not zero every voxel mean. Flattening cannot retain axis interpretation without saved shape/order metadata.

</details>

**Scope:** This local notebook and its numerical checks are part of the executable core. Completion of the external chapter is a learner assignment; its execution is not implied by the local result. No endorsement by the source authors or USC is implied.

### Return to the research question

Reopen [PP01](../../curriculum/papers/processing.md#pp01), [PM05](../../curriculum/papers/modeling.md#pm05) and your initial two-sentence prediction. In your [evidence ledger](../../curriculum/coursework/EVIDENCE_LEDGER.md):

1. Cite one output or diagnostic from this lesson and explain the transformation it demonstrates.
2. Revise one claim or question from the paper, with a figure/section locator. State what this small exercise still cannot establish about the published result.
3. Ask AI to propose a next check. Accept, revise or reject it with a scientific reason. Then explain your decision aloud without reading the AI response.

Reuse this entry in the A2 portfolio when relevant; a separate report is unnecessary.
